# Synthetic Training Data Generator

Generate high-quality synthetic datasets for fine-tuning or evaluation using LLMs and HuggingFace tools.

Given a topic and data type, this tool generates structured synthetic data (Q&A pairs, sentiment samples), analyzes them with HuggingFace tokenizers, and exports to JSON.

**Week 3 Community Contribution by Mirack**

Skills demonstrated: HuggingFace transformers, tokenizers, open source model integration, synthetic data generation.

In [ ]:
import os
import json
from openai import OpenAI
from transformers import AutoTokenizer
import re

In [ ]:

client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENAI_API_KEY")
    
)

MODEL = "gpt-4.1-nano"

In [ ]:
# Load HuggingFace tokenizer to analyze generated data
tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"Loaded tokenizer: {tokenizer.name_or_path}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

## 1. Generate Synthetic Q&A Pairs

In [ ]:
def generate_qa_pairs(topic, num_pairs=5, difficulty="mixed"):
    """Generate synthetic Q&A pairs for a given topic."""
    prompt = f"""Generate exactly {num_pairs} question-answer pairs about: {topic}
Difficulty level: {difficulty}

Respond ONLY with a valid JSON array, no markdown, no extra text:
[
  {{"question": "...", "answer": "...", "difficulty": "easy|medium|hard"}}
]

Make questions diverse - mix factual, conceptual, and applied questions."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw)

In [ ]:
qa_data = generate_qa_pairs("Python decorators", num_pairs=5)

for i, pair in enumerate(qa_data, 1):
    print(f"Q{i}: {pair['question']}")
    print(f"A{i}: {pair['answer']}")
    print(f"    Difficulty: {pair['difficulty']}\n")

## 2. Generate Sentiment Analysis Data

In [ ]:
def generate_sentiment_data(domain, num_samples=10):
    """Generate synthetic sentiment-labeled text data."""
    prompt = f"""Generate exactly {num_samples} realistic text samples for sentiment analysis in the domain: {domain}

Respond ONLY with a valid JSON array, no markdown:
[
  {{"text": "...", "sentiment": "positive|negative|neutral", "confidence": 0.0-1.0}}
]

Balance the sentiments roughly equally. Make texts realistic and varied in length."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw)

In [ ]:
sentiment_data = generate_sentiment_data("restaurant reviews", num_samples=6)

for i, sample in enumerate(sentiment_data, 1):
    print(f"[{sample['sentiment'].upper():>8}] ({sample['confidence']:.1f}) {sample['text']}\n")

## 3. Tokenizer Analysis on Generated Data

In [ ]:
def analyze_dataset_tokens(dataset, text_field="text"):
    """Analyze token statistics of a dataset using HuggingFace tokenizer."""
    if text_field == "qa":
        texts = [f"{item['question']} {item['answer']}" for item in dataset]
    else:
        texts = [item[text_field] for item in dataset]
    
    token_counts = []
    for text in texts:
        tokens = tokenizer.encode(text)
        token_counts.append(len(tokens))
    
    total_tokens = sum(token_counts)
    avg_tokens = total_tokens / len(token_counts)
    
    print(f"Dataset size: {len(texts)} samples")
    print(f"Total tokens: {total_tokens:,}")
    print(f"Avg tokens/sample: {avg_tokens:.1f}")
    print(f"Min tokens: {min(token_counts)}")
    print(f"Max tokens: {max(token_counts)}")
    
    first_tokens = tokenizer.encode(texts[0])
    decoded = [tokenizer.decode([t]) for t in first_tokens[:15]]
    print(f"\nSample tokenization (first entry):")
    print(f"  First 15 tokens: {decoded}")
    
    return {"total": total_tokens, "avg": avg_tokens, "counts": token_counts}

In [ ]:
print("=== Q&A Dataset Token Analysis ===")
qa_stats = analyze_dataset_tokens(qa_data, text_field="qa")

print("\n=== Sentiment Dataset Token Analysis ===")
sent_stats = analyze_dataset_tokens(sentiment_data, text_field="text")

## 4. Export to Fine-tuning Format

In [ ]:
def export_as_finetune_jsonl(qa_pairs):
    """Convert Q&A pairs to OpenAI fine-tuning JSONL format."""
    finetune_data = []
    for pair in qa_pairs:
        entry = {
            "messages": [
                {"role": "user", "content": pair["question"]},
                {"role": "assistant", "content": pair["answer"]}
            ]
        }
        finetune_data.append(entry)
    return finetune_data

finetune_ready = export_as_finetune_jsonl(qa_data)
print(json.dumps(finetune_ready[0], indent=2))
print(f"\nTotal fine-tuning samples: {len(finetune_ready)}")

## 5. Bulk Generation

In [ ]:
def generate_bulk_dataset(topics, pairs_per_topic=3):
    """Generate a larger dataset across multiple topics."""
    all_pairs = []
    for topic in topics:
        print(f"Generating data for: {topic}")
        pairs = generate_qa_pairs(topic, num_pairs=pairs_per_topic)
        for pair in pairs:
            pair["topic"] = topic
        all_pairs.extend(pairs)
    print(f"\nTotal samples generated: {len(all_pairs)}")
    return all_pairs

topics = ["Python list comprehensions", "REST API design", "Docker containers"]
bulk_data = generate_bulk_dataset(topics, pairs_per_topic=3)

print("\n=== Bulk Dataset Token Analysis ===")
bulk_stats = analyze_dataset_tokens(bulk_data, text_field="qa")